In [11]:
!pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
import pandas as pd

columns = ['Class','age','menopause','tumor_size','inv_nodes',
           'node_caps','deg_malig','breast','breast_quad','irradiat']

df = pd.read_csv('breast-cancer.data', names=columns)
print(df.head())


                  Class    age menopause tumor_size inv_nodes node_caps  \
0  no-recurrence-events  30-39   premeno      30-34       0-2        no   
1  no-recurrence-events  40-49   premeno      20-24       0-2        no   
2  no-recurrence-events  40-49   premeno      20-24       0-2        no   
3  no-recurrence-events  60-69      ge40      15-19       0-2        no   
4  no-recurrence-events  40-49   premeno        0-4       0-2        no   

   deg_malig breast breast_quad irradiat  
0          3   left    left_low       no  
1          2  right    right_up       no  
2          2   left    left_low       no  
3          2  right     left_up       no  
4          2  right   right_low       no  


In [13]:
# Import required libraries
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Load scikit-learn Breast Cancer dataset
data = load_breast_cancer(as_frame=True)
X = data.data  # Features (30 columns)
y = data.target  # Target (0=Malignant, 1=Benign)

print("Dataset Info:")
print(f"Shape: {X.shape} (569 samples, 30 features)")
print(f"Features: {X.shape[1]}")
print(f"Classes: {len(np.unique(y))}")
print("\nTarget Distribution:")
print(pd.Series(y).value_counts().sort_index())
print(f"\nFeature names (first 5): {data.feature_names[:5]}")


Dataset Info:
Shape: (569, 30) (569 samples, 30 features)
Features: 30
Classes: 2

Target Distribution:
target
0    212
1    357
Name: count, dtype: int64

Feature names (first 5): ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness']


In [14]:
# Split: 80% train, 20% test (stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # Reproducible results
    stratify=y          # Maintain class proportions
)

print("✅ Split Complete!")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTrain class distribution:\n{pd.Series(y_train).value_counts().sort_index()}")
print(f"Test class distribution:\n{pd.Series(y_test).value_counts().sort_index()}")


✅ Split Complete!
Training set: 455 samples
Test set:     114 samples

Train class distribution:
target
0    170
1    285
Name: count, dtype: int64
Test class distribution:
target
0    42
1    72
Name: count, dtype: int64


In [15]:
# Import modeling libraries
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scale features (CRITICAL for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled!")
print(f"Training features mean: {X_train_scaled.mean():.3f} (should be ~0)")
print(f"Training features std:  {X_train_scaled.std():.3f} (should be ~1)")


✅ Features scaled!
Training features mean: 0.000 (should be ~0)
Training features std:  1.000 (should be ~1)


In [16]:
# Train Logistic Regression
lr_model = LogisticRegression(
    random_state=42, 
    max_iter=1000  # Ensure convergence
)

lr_model.fit(X_train_scaled, y_train)

print("✅ Logistic Regression trained!")
print(f"Model coefficients shape: {lr_model.coef_.shape}")
print(f"Training accuracy: {lr_model.score(X_train_scaled, y_train):.3f}")


✅ Logistic Regression trained!
Model coefficients shape: (1, 30)
Training accuracy: 0.989


/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_fea

In [17]:
# Predictions for classification report
y_pred = lr_model.predict(X_test_scaled)

# Probability scores for Lift metric  
y_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]  # Probability of Benign (class 1)

print("✅ Predictions complete!")
print(f"Test predictions shape: {y_pred.shape}")
print(f"Prediction probabilities shape: {y_pred_proba.shape}")
print(f"Sample predictions: {y_pred[:5]}")
print(f"Sample probabilities: {[f'{p:.3f}' for p in y_pred_proba[:5]]}")  # ✅ FIXED
print(f"Test accuracy: {lr_model.score(X_test_scaled, y_test):.3f}")


✅ Predictions complete!
Test predictions shape: (114,)
Prediction probabilities shape: (114,)
Sample predictions: [0 1 0 1 0]
Sample probabilities: ['0.000', '1.000', '0.006', '0.534', '0.000']
Test accuracy: 0.982


/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ashna/.pyenv/versions/3.13.5/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/ashna/.pyenv/versions/3.13.5/lib/py

In [18]:
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
import pandas as pd

# 3.1 Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("\nWhere:")
print("[[TN, FP]") 
print(" [FN, TP]]")
print("TN=True Neg (Malignant correct), FP=False Pos, FN=False Neg, TP=True Pos (Benign correct)")


Confusion Matrix:
[[41  1]
 [ 1 71]]

Where:
[[TN, FP]
 [FN, TP]]
TN=True Neg (Malignant correct), FP=False Pos, FN=False Neg, TP=True Pos (Benign correct)


In [19]:
# 3.2 Classification metrics
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
accuracy = (y_pred == y_test).mean()

print(f"Classification Rate (Accuracy):  {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"Precision:                      {precision:.3f}")
print(f"Recall:                         {recall:.3f}")
print(f"F1-Score:                       {f1:.3f}")

# Detailed classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))


Classification Rate (Accuracy):  0.982 (98.2%)
Precision:                      0.986
Recall:                         0.986
F1-Score:                       0.986

Detailed Classification Report:
              precision    recall  f1-score   support

   Malignant       0.98      0.98      0.98        42
      Benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [20]:
# 3.3 Lift Chart (Top 10% of predictions)
top_10_percent = int(len(y_test) * 0.1)  # Top 10% = ~11 samples
top_indices = np.argsort(y_pred_proba)[::-1][:top_10_percent]  # Highest probability scores

# Actual positives in top 10%
actual_positives_top10 = y_test.iloc[top_indices].sum()
total_top10 = len(top_indices)

# Expected positives if random
expected_positives_top10 = total_top10 * y_test.mean()

# Lift = actual / expected
lift = actual_positives_top10 / expected_positives_top10 if expected_positives_top10 > 0 else 0

print(f"\n🎯 LIFT METRIC (Top 10% predictions):")
print(f"Top 10% samples:     {total_top10}")
print(f"Actual Benign cases: {actual_positives_top10}")
print(f"Expected (random):   {expected_positives_top10:.1f}")
print(f"Lift ratio:          {lift:.2f}x")



🎯 LIFT METRIC (Top 10% predictions):
Top 10% samples:     11
Actual Benign cases: 11
Expected (random):   6.9
Lift ratio:          1.58x


In [21]:
# Summary table
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Lift (Top 10%)'],
    'Value': [f'{accuracy:.3f}', f'{precision:.3f}', f'{recall:.3f}', f'{f1:.3f}', f'{lift:.2f}x'],
    'Interpretation': [
        'Overall correct predictions',
        "When model says 'Benign', it's correct X% of time", 
        "Model catches X% of actual Benign cases",
        'Balance of Precision + Recall',
        'Top 10% predictions are X times better than random'
    ]
})

print("\n📊 FINAL METRICS SUMMARY")
print("=" * 60)
print(results_df.to_string(index=False))



📊 FINAL METRICS SUMMARY
        Metric Value                                     Interpretation
      Accuracy 0.982                        Overall correct predictions
     Precision 0.986  When model says 'Benign', it's correct X% of time
        Recall 0.986            Model catches X% of actual Benign cases
      F1-Score 0.986                      Balance of Precision + Recall
Lift (Top 10%) 1.58x Top 10% predictions are X times better than random
